In [ ]:
import os
import sys

# Step 1: Set CUDA_VISIBLE_DEVICES=0 before importing PyTorch
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(f"CUDA_VISIBLE_DEVICES set to: {os.environ.get('CUDA_VISIBLE_DEVICES')}")

In [ ]:
import subprocess
import hashlib
import json
from datetime import datetime, timezone

# Target commit SHA
EXECUTED_CODE_SHA = "502871a57b275bffaa65ba6a24f9e834c3721fb6"
REPO_URL = "https://github.com/ibn-e-Muhammad/fq-ssiuu.git"
WORKING_DIR = "/kaggle/working"
REPO_DIR = "/kaggle/working/fq-ssiuu"
OUTPUT_DIR = "/kaggle/working/og001_output"

print(f"Target EXECUTED_CODE_SHA: {EXECUTED_CODE_SHA}")

In [ ]:
# Step 2 & 3: Clone public repository and checkout EXECUTED_CODE_SHA
if os.path.exists(REPO_DIR):
    print(f"Repository directory {REPO_DIR} already exists.")
else:
    print(f"Cloning public repository {REPO_URL} into {REPO_DIR}...")
    clone_res = subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        capture_output=True, text=True
    )
    print(f"Clone stdout: {clone_res.stdout}")
    print(f"Clone stderr: {clone_res.stderr}")
    if clone_res.returncode != 0:
        raise RuntimeError(f"Git clone failed with return code {clone_res.returncode}")

print(f"Checking out commit {EXECUTED_CODE_SHA}...")
checkout_res = subprocess.run(
    ["git", "checkout", EXECUTED_CODE_SHA],
    capture_output=True, text=True, cwd=REPO_DIR
)
print(f"Checkout stdout: {checkout_res.stdout}")
print(f"Checkout stderr: {checkout_res.stderr}")
if checkout_res.returncode != 0:
    raise RuntimeError(f"Git checkout failed with return code {checkout_res.returncode}")

In [ ]:
# Step 4: Verify checked-out HEAD equals EXECUTED_CODE_SHA
rev_res = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    capture_output=True, text=True, cwd=REPO_DIR
)
actual_head = rev_res.stdout.strip()
print(f"Actual checked-out HEAD: {actual_head}")

if actual_head != EXECUTED_CODE_SHA:
    raise RuntimeError(f"HEAD mismatch! Expected {EXECUTED_CODE_SHA}, observed {actual_head}")
else:
    print("Checked-out HEAD verified successfully.")

In [ ]:
# Step 5: Record orchestration notebook SHA-256 separately from executed repository SHA
nb_path = "/kaggle/working/OG001_KAGGLE_OVERLAY_VALIDATION.ipynb"
notebook_sha256 = "UNAVAILABLE"
if os.path.exists(nb_path):
    h = hashlib.sha256()
    with open(nb_path, "rb") as f:
        while chunk := f.read(65536):
            h.update(chunk)
    notebook_sha256 = h.hexdigest()

print(f"Orchestration notebook SHA-256: {notebook_sha256}")

In [ ]:
# Step 6 & 7: Run scripts/kaggle_setup.py --apply --output-dir /kaggle/working/og001_output
env = dict(os.environ)
env["CUDA_VISIBLE_DEVICES"] = "0"
env["PYTHONPATH"] = REPO_DIR

setup_cmd = [
    sys.executable,
    os.path.join(REPO_DIR, "scripts/kaggle_setup.py"),
    "--apply",
    "--output-dir", OUTPUT_DIR
]

print("Executing kaggle_setup.py --apply...")
res_setup = subprocess.run(
    setup_cmd,
    capture_output=True,
    text=True,
    cwd=REPO_DIR,
    env=env
)

print(f"Setup return code: {res_setup.returncode}")
print(f"Setup stdout:\n{res_setup.stdout}")
print(f"Setup stderr:\n{res_setup.stderr}")

# Save stdout and stderr logs in WORKING_DIR
with open(os.path.join(WORKING_DIR, "kaggle_runner_stdout.log"), "w", encoding="utf-8") as f:
    f.write(res_setup.stdout)
with open(os.path.join(WORKING_DIR, "kaggle_runner_stderr.log"), "w", encoding="utf-8") as f:
    f.write(res_setup.stderr)

In [ ]:
# Step 8, 9, 10: Always reach final artifact-collection cell and generate kernel_summary.json
summary = {
    "schema_version": "1.0",
    "execution_utc": datetime.now(timezone.utc).isoformat(),
    "target_executed_code_sha": EXECUTED_CODE_SHA,
    "actual_repo_head": actual_head,
    "head_matches": (actual_head == EXECUTED_CODE_SHA),
    "notebook_sha256": notebook_sha256,
    "setup_returncode": res_setup.returncode,
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "output_dir": OUTPUT_DIR,
    "setup_status_file_exists": os.path.exists(os.path.join(OUTPUT_DIR, "setup_status.json")),
    "manifest_file_exists": os.path.exists(os.path.join(OUTPUT_DIR, "MANIFEST.json")),
    "overlay_lock_exists": os.path.exists(os.path.join(OUTPUT_DIR, "kaggle-overlay.lock"))
}

status_file = os.path.join(OUTPUT_DIR, "setup_status.json")
if os.path.exists(status_file):
    try:
        with open(status_file, "r", encoding="utf-8") as f:
            summary["setup_status_payload"] = json.load(f)
    except Exception as e:
        summary["setup_status_error"] = str(e)

summary_path = os.path.join(WORKING_DIR, "kernel_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("kernel_summary.json created successfully:")
print(json.dumps(summary, indent=2))